In [2]:
%pip install ipycanvas

In [4]:
import asyncio, random
from ipycanvas import Canvas, hold_canvas
W, H = 400, 300
cv = Canvas(width=W, height=H)
pa_o, pa_a, pe_r = 75, 12, 6
px, py = (W - pa_o) / 2, H - 25
cx, cy, dx, dy = W/2, H/2, 3, -3
las, pdr, v_b, v_l, frq, t_p, t_pd, vds, rnd, dif, est, tsk, f_c = [], [], 2.5, 2.5, 0.004, None, 0, 3, 1, "Fácil", "INICIO", None, 0

# Las misiones ahora mantienen su progreso entre partidas
mis = {
    "m1": {"tx": "Romper 10 bloques", "ac": 0, "me": 10},
    "m2": {"tx": "Llegar a la Ronda 2", "ac": 1, "me": 2},
    "m3": {"tx": "Atrapar 2 poderes", "ac": 0, "me": 2}
}
CLS = ['#ff4757', '#ffa502', '#eccc68', '#1e90ff']

def c_blq():
    return [{'x': c*60+25, 'y': f*20+30, 'w': 52, 'h': 14, 'p': random.random()<0.15, 'c': CLS[f]} for f in range(4) for c in range(6)]
blq = c_blq()

def scr_i():
    with hold_canvas(cv):
        cv.clear(); cv.fill_style = '#1e272e'; cv.fill_rect(0, 0, W, H)
        cv.fill_style = '#ffa502'; cv.font = 'bold 26px sans-serif'; cv.fill_text("BLOCK BREAKER", 70, H/2 - 50)
        cv.fill_style = '#2ecc71'; cv.font = 'bold 18px sans-serif'; cv.fill_text("LASER EDITION", 125, H/2 - 20)
        
        # Botón JUGAR (Verde)
        cv.fill_style = '#2ed573'; cv.fill_rect(60, H/2 + 20, 120, 35)
        cv.fill_style = '#1e272e'; cv.font = 'bold 12px sans-serif'; cv.fill_text("🎮 JUGAR", 95, H/2 + 42)
        
        # Botón MISIONES (Amarillo)
        cv.fill_style = '#eccc68'; cv.fill_rect(220, H/2 + 20, 120, 35)
        cv.fill_style = '#1e272e'; cv.font = 'bold 12px sans-serif'; cv.fill_text("🏆 MISIONES", 245, H/2 + 42)

def scr_misiones():
    with hold_canvas(cv):
        cv.clear(); cv.fill_style = '#2f3542'; cv.fill_rect(0, 0, W, H)
        cv.fill_style = 'gold'; cv.font = 'bold 20px sans-serif'; cv.fill_text("🏆 OBJETIVOS", W/2 - 60, 40)
        cv.stroke_style = '#57606f'; cv.stroke_line(30, 55, W - 30, 55)
        
        y = 95
        for k, m in mis.items():
            ok = m["ac"] >= m["me"]
            cv.fill_style = '#2ed573' if ok else '#ffffff'
            cv.font = '13px sans-serif'
            cv.fill_text(f"{'✅' if ok else '⬜'} {m['tx']}", 40, y)
            cv.fill_style = '#a4b0be'; cv.font = '11px sans-serif'
            cv.fill_text(f"Progreso: {min(m['ac'], m['me'])}/{m['me']}", 65, y + 18)
            y += 55
            
        # Botón VOLVER (Blanco)
        cv.fill_style = '#ffffff'; cv.fill_rect(140, H - 45, 120, 30)
        cv.fill_style = '#1e272e'; cv.font = 'bold 12px sans-serif'; cv.fill_text("⬅️ VOLVER", 175, H - 25)

def scr_g(mg):
    with hold_canvas(cv):
        cv.clear(); cv.fill_style = '#1e272e'; cv.fill_rect(0, 0, W, H)
        cv.fill_style = '#ff4757'; cv.font = 'bold 24px sans-serif'; cv.fill_text("GAME OVER", W/2 - 65, H/2 - 30)
        cv.fill_style = 'white'; cv.font = '13px sans-serif'; cv.fill_text(mg, W/2 - 80, H/2 + 5)
        cv.fill_style = '#eccc68'; cv.font = '11px sans-serif'; cv.fill_text("Haz click para continuar", W/2 - 55, H/2 + 45)

@cv.on_mouse_down
def al_ck(x, y):
    global est, vds, rnd, dif, blq, las, pdr, tsk, cx, cy, dx, dy
    
    if est == "INICIO":
        # Click en JUGAR (X de 60 a 180, Y de 170 a 205)
        if 60 <= x <= 180 and H/2 + 20 <= y <= H/2 + 55:
            if tsk: tsk.cancel()
            vds, rnd, dif = 3, 1, "Fácil"
            blq = c_blq(); las.clear(); pdr.clear()
            cx, cy, dx, dy = W / 2, H / 2, 3, -3
            mis["m2"]["ac"] = 1
            est = "JUGANDO"
            tsk = asyncio.ensure_future(b_fis())
        # Click en MISIONES (X de 220 a 340, Y de 170 a 205)
        elif 220 <= x <= 340 and H/2 + 20 <= y <= H/2 + 55:
            est = "PANTALLA_MISIONES"
            scr_misiones()
            
    elif est == "PANTALLA_MISIONES":
        # Click en VOLVER (X de 140 a 260, Y de 255 a 285)
        if 140 <= x <= 260 and H - 45 <= y <= H - 15:
            est = "INICIO"
            scr_i()
            
    elif est == "GAME_OVER":
        # Al perder, cualquier click te devuelve a la pantalla de inicio
        est = "INICIO"
        scr_i()

@cv.on_mouse_move
def m_pal(x, y):
    global px
    if est == "JUGANDO":
        px = x - pa_o / 2
        if px < 0: px = 0
        if px > W - pa_o: px = W - pa_o

async def b_fis():
    global cx, cy, dx, dy, vds, las, blq, rnd, frq, v_l, v_b, dif, pdr, t_pd, t_p, pa_o, est, f_c
    try:
        while est == "JUGANDO":
            f_c += 1
            if t_pd > 0:
                t_pd -= 1
                if t_pd == 0: pa_o, v_l, t_p = 75, v_b, None
            cx += dx; cy += dy
            if cx - pe_r <= 0 or cx + pe_r >= W: dx *= -1
            if cy - pe_r <= 0: dy *= -1
            if cy + pe_r >= H:
                vds -= 1; las.clear(); pdr.clear()
                if vds > 0: cx, cy, dy = W / 2, H / 2, -3
                else: est = "GAME_OVER"; scr_g(f"Alcanzaste la Ronda {rnd}"); break
            if py <= cy + pe_r <= py + pa_a and px <= cx <= px + pa_o: dy = -abs(dy)
            for b in blq[:]:
                if b['x'] <= cx <= b['x'] + b['w'] and b['y'] <= cy <= b['y'] + b['h']:
                    mis["m1"]["ac"] += 1
                    if b['p']: pdr.append({'x': b['x'] + b['w']/2, 'y': b['y'], 'r': 6})
                    blq.remove(b); dy *= -1; break
            if random.random() < frq: las.append({'x': random.randint(10, W - 10), 'y': 0, 'w': 4, 'h': 15})
            for l in las[:]:
                l['y'] += v_l
                if py <= l['y'] + l['h'] <= py + pa_a and px <= l['x'] <= px + pa_o:
                    vds -= 1; las.clear(); pdr.clear()
                    if vds > 0: cx, cy, dy = W / 2, H / 2, -3; break
                    else: est = "GAME_OVER"; scr_g("¡Derrotado por Láser!"); break
                if l['y'] > H: las.remove(l)
            if est == "GAME_OVER": break
            for p in pdr[:]:
                p['y'] += 2
                if py <= p['y'] + p['r'] <= py + pa_a and px <= p['x'] <= px + pa_o:
                    mis["m3"]["ac"] += 1
                    st = random.choice(["G", "R", "V"])
                    if st == "G": pa_o, t_pd, t_p = 130, 400, "Paleta Gigante"
                    elif st == "R": v_l, t_pd, t_p = v_b * 0.5, 400, "Láseres Lentos"
                    elif st == "V": vds, t_pd, t_p = vds + 1, 100, "+1 Vida Obtenida"
                    pdr.remove(p)
                elif p['y'] > H: pdr.remove(p)
            if not blq:
                rnd += 1; mis["m2"]["ac"] = rnd
                frq += 0.003; v_b += 0.4; v_l = v_b
                dy = dy - 0.2 if dy < 0 else dy + 0.2
                dif = "Normal" if rnd == 2 else "Intermedio" if rnd == 3 else "Difícil"
                blq, las, pdr = c_blq(), [], []
                cx, cy = W / 2, H / 2
                with hold_canvas(cv):
                    cv.clear(); cv.fill_style = '#2f3542'; cv.fill_rect(0, 0, W, H)
                    cv.fill_style = 'gold'; cv.font = '24px sans-serif'; cv.fill_text(f"¡Ronda {rnd}!", W/2 - 50, H/2 - 20)
                await asyncio.sleep(2.5)
            with hold_canvas(cv):
                cv.clear()
                cv.fill_style = '#7ed6df'; cv.fill_rect(0, 0, W, H - 120)
                cv.fill_style = '#2ed573'; cv.fill_rect(0, H - 120, W, 120)
                cv.fill_style = '#2f3542'; cv.font = 'bold 14px sans-serif'
                cv.fill_text(f"Vidas: {vds}", 15, 20); cv.fill_text(f"Ronda: {rnd}", 145, 20); cv.fill_text(f"Dificultad: {dif}", 250, 20)
                if t_p: cv.fill_style = '#b33939'; cv.font = 'bold 12px sans-serif'; cv.fill_text(f"⚡ {t_p}", 15, 42)
                cv.fill_style = '#ffffff'; cv.fill_rect(px, py, pa_o, pa_a)
                cv.fill_style = '#2f3542'; cv.fill_arc(cx, cy, pe_r, 0, 6.28); cv.fill()
                for b in blq:
                    cv.fill_style = '#ffffff' if b['p'] and (f_c // 10) % 2 == 0 else b['c']
                    cv.fill_rect(b['x'], b['y'], b['w'], b['h'])
                cv.fill_style = '#ff4757'
                for l in las: cv.fill_rect(l['x'], l['y'], l['w'], l['h'])
                for p in pdr:
                    cv.fill_style = '#ffa502'; cv.fill_arc(p['x'], p['y'], p['r'], 0, 6.28); cv.fill()
                    cv.fill_style = 'white'; cv.font = 'bold 10px sans-serif'; cv.fill_text("P", p['x'] - 3, p['y'] + 3)
            await asyncio.sleep(0.016)
    except asyncio.CancelledError: pass

display(cv)
scr_i()


Canvas(height=300, width=400)